# RSX-V2-15 · Locked prospective confirmation (T4 GPU)

**Requires RSX-V2-14 to have completed and locked its policy.** 60 new, template-disjoint held-out problems × (source + finalize + remask + regenerate) = **240 planned model calls**. Preserve is computed from source with no extra generation. Uses immutable policy hash and data manifest; individual trials are saved to a separate Drive folder. Does not change the policy based on test results. Automatically stops if V2-14 fails calibration.

Run after V2-14, not concurrently. Re-run this notebook to resume unfinished trials without repeating completed calls.

In [ ]:

import os,sys,subprocess
os.environ['HF_HUB_DISABLE_XET']='1'
os.environ['TOKENIZERS_PARALLELISM']='false'
CODE_COMMIT='fe225a99e0b5aeff351804e871d7581155d7f27d'
REPO_DIR='/content/ALoDLM_RSX_E10E11'
if not os.path.isdir(os.path.join(REPO_DIR,'.git')):
    subprocess.run(['git','clone','-q','https://github.com/amazon-science/ALoDLM.git',REPO_DIR],check=True)
subprocess.run(['git','-C',REPO_DIR,'checkout','--detach','-q',CODE_COMMIT],check=True)
assert subprocess.check_output(['git','-C',REPO_DIR,'rev-parse','HEAD'],text=True).strip()==CODE_COMMIT
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.56.1','accelerate==1.10.1','huggingface-hub>=0.34,<1.0','safetensors>=0.6','PyYAML>=6'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',REPO_DIR,'--no-deps'],check=True)
print('Pinned ALoDLM source:',CODE_COMMIT)


In [ ]:
import json,os,re,time,random,math,ast,hashlib,zipfile,zlib,base64,gc,csv,sys
from pathlib import Path
from collections import Counter,defaultdict
from types import SimpleNamespace
import numpy as np,torch,pandas as pd
from huggingface_hub import snapshot_download
from google.colab import drive,files
from IPython.display import display
if not torch.cuda.is_available(): raise RuntimeError('A Colab GPU (T4 or better) is required')
if torch.cuda.get_device_properties(0).total_memory<12*2**30: raise RuntimeError('At least 12 GiB GPU VRAM required')
drive.mount('/content/drive')
MODEL_ID='amazon/ALoDLM-1.7B';MODEL_REV='588c7dc1946e3e09fd496da0b0808bd26f46a758'
Q=0.4;TAU=0.4;WINDOW=16
PRIOR_PROMPT='\n\nGive a short solution. End with one line: FINAL_ANSWER: <number>. For Countdown include an expression using all inputs once.'
DIRECT_PROMPT='\n\nDo not provide working. Reply with one line containing the final numerical answer. For Countdown provide a valid expression before the final number.'
FRESH_PROMPT='\n\nIndependently solve the problem from the beginning. Verify any computation. End with FINAL_ANSWER: <number>. For Countdown give a valid expression using each number once.'
def digest(v):
    return hashlib.sha256(json.dumps(v,sort_keys=True,separators=(',',':'),ensure_ascii=False).encode()).hexdigest()
def save_new_json(path,obj):
    if path.exists():
        old=json.loads(path.read_text());assert old==obj,'Immutable protocol/data mismatch at '+str(path)
    else:
        temp=path.with_name(path.name+'.pending');temp.write_text(json.dumps(obj,indent=2,sort_keys=True,ensure_ascii=False));os.replace(temp,path)
def atomic_record(obj):
    assert obj['protocol_hash']==PROTOCOL_HASH
    tid=obj['trial_id'];p=RECS/(hashlib.sha256(tid.encode()).hexdigest()+'.json')
    if p.exists():assert json.loads(p.read_text())==obj,'Conflicting trial ID '+tid
    else:
        t=p.with_suffix('.pending');t.write_text(json.dumps(obj,sort_keys=True,indent=2,ensure_ascii=False));os.replace(t,p)
def existing(tid):
    p=RECS/(hashlib.sha256(tid.encode()).hexdigest()+'.json')
    if not p.exists():return None
    obj=json.loads(p.read_text());assert obj['trial_id']==tid and obj['protocol_hash']==PROTOCOL_HASH
    return obj
def read_records():
    out=[json.loads(f.read_text()) for f in sorted(RECS.glob('*.json'))]
    assert len(out)==len({r['trial_id'] for r in out})
    assert all(r['protocol_hash']==PROTOCOL_HASH for r in out)
    return out
def set_seed(seed):
    random.seed(seed);np.random.seed(seed%(2**32-1));torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)
def make_prompt(content):
    return list(tokenizer.apply_chat_template([{'role':'user','content':content}],tokenize=True,add_generation_prompt=True,enable_thinking=False))
def render(ids):return tokenizer.decode(list(map(int,ids)),skip_special_tokens=True).strip()
def model_call(prompt,prefix,mode,max_new,seed):
    set_seed(seed)
    cfg=DecodeConfig(q=Q,tau=TAU,mode=mode,window_size=WINDOW,max_new_tokens=int(max_new),position_penalty=None)
    cfg.validate()
    torch.cuda.reset_peak_memory_stats();start=time.monotonic()
    out=decoder.generate(make_prompt(prompt)+list(map(int,prefix)),cfg)
    new=list(map(int,out['token_ids']))
    return {'out':render(list(prefix)+new),'ids':list(prefix)+new,'new_ids':new,
            'generated_tokens':int(out.get('generated_tokens',len(new))),
            'loop_token_passes':int(out.get('loop_token_passes',0)),
            'outer_steps':int(out.get('outer_steps',0)),
            'stop_reason':str(out.get('stop_reason','unknown')),
            'wall_seconds':round(time.monotonic()-start,3),
            'peak_vram_gb':round(float(torch.cuda.max_memory_allocated()/2**30),3)}
def seed_for(pid):return 602600 + int(hashlib.sha256(pid.encode()).hexdigest()[:6],16) % 999999

def character_cut(ids,offset):
    for i in range(1,len(ids)):
        if len(render(ids[:i]))>=offset:return max(1,i-1)
    return max(1,len(ids)//2)
def repair_cut(ids,action):
    n=len(ids)
    if n<2:return 0
    if action=='first_math':
        m=re.search(r'(?:\d+\s*[+*/-]\s*\d+)|(?:\d+\s*=)',render(ids))
        return min(n-1,max(1,character_cut(ids,m.start()) if m else n//2))
    if action=='tail50':return min(n-1,max(1,n//2))
    raise ValueError(action)
def snapshot_export():
    archive=Path('/content')/(STUDY+'_Checkpoint.zip')
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
        for p in sorted(ROOT.rglob('*')):
            if p.is_file() and not p.name.endswith('.pending'):z.write(p,p.relative_to(ROOT))
    print('Archive',archive,'size',archive.stat().st_size,'saved records',len(read_records()))
    try:files.download(str(archive))
    except Exception as exc:print('Auto-download failed; file remains at',archive,repr(exc))
    return archive
FINAL_RE=re.compile(r'(?im)^\s*FINAL_ANSWER\s*:\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)\s*$')
ANSWER_RE=re.compile(r'(?i)(?:the\s+)?(?:final\s+)?answer\s*(?:is|:)\s*([-+]?\d+(?:\.\d+)?)')
NUM_RE=re.compile(r'[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?')
RUN_RE=re.compile(r'[0-9()+*×÷/\-+=\s]{3,}')
def closenum(a,b):return a is not None and b is not None and math.isclose(float(a),float(b),rel_tol=1e-9,abs_tol=1e-8)
def safe_expression(s):
    try:t=ast.parse(s,mode='eval')
    except Exception:return None
    lit=[]
    def walk(n):
        if isinstance(n,ast.Expression):return walk(n.body)
        if isinstance(n,ast.Constant) and type(n.value) in (int,float):lit.append(float(n.value));return float(n.value)
        if isinstance(n,ast.UnaryOp) and isinstance(n.op,(ast.UAdd,ast.USub)):
            a=walk(n.operand);return (-a if isinstance(n.op,ast.USub) else a)
        if isinstance(n,ast.BinOp) and isinstance(n.op,(ast.Add,ast.Sub,ast.Mult,ast.Div)):
            a,b=walk(n.left),walk(n.right)
            if isinstance(n.op,ast.Add):return a+b
            if isinstance(n.op,ast.Sub):return a-b
            if isinstance(n.op,ast.Mult):return a*b
            if abs(b)<1e-12:raise ZeroDivisionError
            return a/b
        raise ValueError('Unsupported expression')
    try:
        v=walk(t)
        if math.isfinite(v):return {'value':v,'literals':lit}
    except Exception:pass
    return None

def extract_expression(text,problem):
    cleaned=text.replace('×','*').replace('÷','/')
    for m in RUN_RE.finditer(cleaned):
        raw=m.group(0)
        # Restrict to expression strings and avoid accidentally matching a target alone.
        for part in raw.split('='):
            ex=part.strip()
            if not any(ch in ex for ch in '+-*/'):continue
            result=safe_expression(ex)
            if not result:continue
            if len(result['literals'])==len(problem['numbers']) and sorted(result['literals'])==sorted(map(float,problem['numbers'])) and closenum(result['value'],problem['target']):return ex
    return None

BOX_RE=re.compile(r'\\boxed\s*\{\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+))\s*\}')
FINAL_RE=re.compile(r'(?im)^\s*FINAL_ANSWER\s*:\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)\s*$')
ANSWER_RE=re.compile(r'(?i)\b(?:the\s+)?(?:final\s+)?answer\s*(?:is|:)\s*([-+]?\d+(?:\.\d+)?)')

def last_explicit_number(text):
    """Only explicit answer markers, never final arbitrary numerals in unfinished reasoning.
    Conflicting explicit answer values => ambiguous, never an eligible failure."""
    text=str(text)
    entries=[]
    for pattern,kind in ((FINAL_RE,'final'),(BOX_RE,'boxed'),(ANSWER_RE,'answer_phrase')):
        for m in pattern.finditer(text):
            try:entries.append((m.start(),float(m.group(1)),kind))
            except (ValueError,OverflowError):pass
    if not entries:return None,'missing'
    unique=[]
    for _,v,_ in entries:
        if not any(closenum(v,w) for w in unique):unique.append(v)
    if len(unique)>1:return None,'conflicting_markers'
    entries.sort()
    return entries[-1][1],entries[-1][2]

def countdown_wrong_candidate(text,p):
    """Inspectable wrong *complete* expression consuming exactly the supplied numbers.
    Do not equate missing expressions with semantic error."""
    for m in RUN_RE.finditer(str(text).replace('×','*').replace('÷','/')):
        for part in m.group(0).split('='):
            ex=part.strip()
            if not any(ch in ex for ch in '+-*/'):continue
            r=safe_expression(ex)
            if r and len(r['literals'])==len(p['numbers']) and sorted(r['literals'])==sorted(map(float,p['numbers'])):
                if not closenum(r['value'],p['target']):return ex
    return None

def grade_output(text,p):
    text=str(text)
    num,origin=last_explicit_number(text)
    sem_numeric=closenum(num,p['target'])
    last_nonempty=next((line.strip() for line in reversed(text.splitlines()) if line.strip()),'')
    fmt=FINAL_RE.fullmatch(last_nonempty)
    strict=bool(fmt and closenum(float(fmt.group(1)),p['target']))
    if p['kind']=='countdown':
        expr=extract_expression(text,p)
        success=expr is not None
        wrong_expr=None if success else countdown_wrong_candidate(text,p)
        explicit_wrong=(num is not None and not sem_numeric)
        # An explicitly wrong answer or an inspectable illegal computation, with no valid solution.
        is_wrong=bool(not success and origin!='conflicting_markers' and (explicit_wrong or wrong_expr is not None))
    else:
        expr=None;wrong_expr=None
        success=bool(sem_numeric)
        is_wrong=bool(num is not None and not sem_numeric and origin!='conflicting_markers')
    return {'success':bool(success),'numeric_correct':bool(sem_numeric),
            'strict_correct':bool(strict),'valid_expression':expr,
            'wrong_expression':wrong_expr,'answer_origin':origin,
            'explicit_final_number':num,'eligible_semantic_failure':bool(is_wrong),
            'failure_type':('definite_wrong' if is_wrong else
                   ('semantically_correct' if success else 'unanswered_or_ambiguous'))}

def public_accept(text,p):
    """Label-blind stopping gate: never consult word-problem gold targets."""
    if p['kind']=='countdown':
        # Target is explicitly present in the Countdown input; a valid expression may be verified.
        return extract_expression(str(text),p) is not None
    num,origin=last_explicit_number(str(text))
    return num is not None and origin!='conflicting_markers'

def features(text,p,n_tokens):
    """Target-free features; no test labels or correctness queries."""
    num,origin=last_explicit_number(text)
    L=int(n_tokens)
    return {'kind':p['kind'],'length_bucket':'short' if L<45 else ('medium' if L<80 else 'long'),
            'has_final':bool(FINAL_RE.search(str(text))), 'answer_kind':origin,
            'repetition':bool(re.search(r'(\b\w+\b)(?:\s+\1){2,}',str(text),re.I))}

def feat_bucket(f):return f['kind']+'|'+('final' if f['has_final'] else 'no_final')

# Essential semantic-failure regression checks (no GPU or network needed):
q={'kind':'arithmetic','target':14,'numbers':[]}
assert grade_output(r'We have \boxed{14}.',q)['success']
assert not grade_output(r'We have \boxed{14}.',q)['eligible_semantic_failure']
assert grade_output('FINAL_ANSWER: 13',q)['eligible_semantic_failure']
assert not grade_output('Then 14 times',q)['eligible_semantic_failure']
assert not grade_output('No answer yet',q)['eligible_semantic_failure']
assert not grade_output('FINAL_ANSWER: 13\n\\boxed{14}',q)['eligible_semantic_failure']
assert grade_output('The answer is 14',q)['success']
p={'kind':'countdown','numbers':[2,3,4],'target':14}
assert grade_output('2 + 3 * 4 = 14',p)['success']
assert not grade_output('2 + 3 * 4 = 14',p)['eligible_semantic_failure']
assert grade_output('2 + 3 + 4 = 9',p)['eligible_semantic_failure']
assert not grade_output('14',p)['eligible_semantic_failure']
assert not grade_output('2 + 4 + 4 = 14',p)['success']
print('Semantic grader regression suite PASSED: correct boxed excluded, explicit wrong admitted, truncations rejected.')

def strict_completed_correct(text,p):
    ev=grade_output(str(text).replace('\\(','').replace('\\)','').replace('\\[','').replace('\\]',''),p)
    if p['kind']!='countdown':
        v,orig=last_explicit_number(str(text))
        return bool(closenum(v,p['target']) and orig!='conflicting_markers')
    if ev['valid_expression'] is None: return False
    # Accept only complete final-answer lines, NOT incomplete math reasoning.
    for match in re.findall(r'(?im)^\s*FINAL_ANSWER\s*:\s*(.*?)\s*$',str(text)):
        raw=match.strip().replace('\\(','').replace('\\)','').replace('\\[','').replace('\\]','')
        try:
            if closenum(float(raw),p['target']):return True
        except (ValueError,TypeError):pass
        exp=safe_expression(raw)
        if exp and closenum(exp['value'],p['target']) and sorted(exp['literals'])==sorted(map(float,p['numbers'])):return True
    return False

def dual_score(text,p):
    main=grade_output(text,p)
    complete=strict_completed_correct(text,p)
    return {'primary':main,'complete_correct':bool(complete),
            'grading_disagreement':bool(main['success'])!=bool(complete),
            'class':('correct' if complete else ('definite_wrong' if main['eligible_semantic_failure'] else 'ambiguous'))}
# Scoring unit tests against prior observed failure modes.
test_cd={'kind':'countdown','target':14,'numbers':[2,3,4]}
assert strict_completed_correct(r'2 + 3 * 4 = 14\nFINAL_ANSWER: 14'.replace('\\n','\n'),test_cd)
assert not strict_completed_correct('2 + 3 * 4 = 14',test_cd)
assert not strict_completed_correct(r'2 + 3 * 4 = 14\nFINAL_ANSWER: \\('.replace('\\n','\n'),test_cd)


# Completed-answer rubric, label-only, may never be invoked by the router itself.
def strict_completed_correct(text,p):
    normalized=str(text).replace(r'\(','').replace(r'\)','').replace(r'\[','').replace(r'\]','')
    if p['kind']!='countdown':
        value, origin=last_explicit_number(normalized)
        return bool(closenum(value,p['target']) and origin!='conflicting_markers')
    if extract_expression(normalized,p) is None:return False
    value,origin=last_explicit_number(normalized)
    return bool(origin!='conflicting_markers' and closenum(value,p['target']))
def dual_score(text,p):
    original=grade_output(text,p)
    strict=strict_completed_correct(text,p)
    return {'primary':original,'complete_correct':strict,'grading_disagreement':bool(original['success'])!=strict,
            'class':('correct' if strict else ('definite_wrong' if original['eligible_semantic_failure'] else 'ambiguous'))}


import re
ACTIONS=['preserve','finalize','remask','regenerate']
ACTIVE=['finalize','remask','regenerate']
def bucket_of(source,kind):
    # source-only observable signature: task family, explicit final marker and stopping reason
    text=source.get('output','')
    has_marker=bool(re.search(r'(?im)^\s*FINAL_ANSWER\s*:',text) or re.search(r'\\boxed\s*\{',text))
    length_stop=source.get('stop_reason','')=='length'
    return kind+'|'+('final' if has_marker else 'no_final')+'|'+('length' if length_stop else 'eos_or_other')
def action_of(policy,source,kind):
    return policy['actions_by_bucket'].get(bucket_of(source,kind),'preserve')
# Defensive test: changing ground truth cannot change bucket or router action
mock={'output':'FINAL_ANSWER: 12','stop_reason':'eos'}
assert bucket_of(mock,'arithmetic')=='arithmetic|final|eos_or_other'
assert bucket_of(mock,'arithmetic')==bucket_of(dict(mock,target=9999),'arithmetic')

STUDY='RSX_V2_15'
ROOT=Path('/content/drive/MyDrive/RepairScopeX/V2_15_Heldout')
RECS=ROOT/'records'; RECS.mkdir(parents=True,exist_ok=True)
MANIFEST=json.loads(zlib.decompress(base64.b64decode('eNqNVE2P2zYQ/SuEztUuSYn6yJ5aJIcCaQ9tURQoCoEih2s2EqmSlDdO0P/eob1rr712UMCQpfeGb4ZvhvxaSJWsd7F4R/4slgARwhaK70hhrJOT/bJ/DzDL+Onw9ggOgkxQ/IWfCkNG/EKFQfnVJZSpuwsibiQXDTJF3WpDmWnaljWtANNTLrUGKYQwzNDGGNkY3nQVZaCaGqqqp2Nby6oRbOz7KlegvAbMNc82JysMcC5k3wMdhQRjKsE6WkPXMt2KjjEhdGt4q8+29I58LcZVP0KWYDwXrCELhyw5gUms+BfBDUzar+nVDmTVd2yUFCpBcQ+6M6OBXgnOWFfJrm8ARqGp5vjrVK1AaybGsReK0Zr3KpcxY6JpsHqvN8sv3t1//9G///hTye7aH04RAbY2ooE5TnSdarVifd1ABbQ3uu4bLelIO9qNmjembiRuOK9erHOghxGMDzA4eEKluE4p9ziFFXJIgIOFM7g0OJ+yJ8VvGyAfGL3/wBiBz8vksYE+7IiDR+zkFiKZPf5j8zXZ8geSNjguBA1CLSRlAIJSZI3IJ0+MTcSmSAJGQLgjP8MTSThHJMG8TKgS74N0j3ml02R19p8VyBL8OMFMfnwfH46Bpbbxb29dIlIFHyNmJ3GZUPyBxJ3DOpJVe+1TFdIR6zQsgA9cOIJTm1mGT3cXY/x/Z+H5DORwtR5Gz4aYhlmmTXG+CPMFv+z2nZSfh6N632Sh6NegLvL2za20Ma16l6FfYJE2/Kr8An+Uv/PihRvSbtl3Lzf6aMbw7COelI0PKUej3+l4SBv6grx4jAN5uASeh76UwabNnMVKWpwOw2uc3cD5Dbx6je9L0f7JncufYHYd5tfhM+0Zx93GBMu59glm12F+Ha72l10K0rq3fr266t54doNj3+D4N7jqkrvw8DrFblP8NvUm14Wn1yl2m+K3qYO/Wwgv193xfsKrpuSUNyWjJa/LLR6K/wC51zlG')))
CASES=json.loads(zlib.decompress(base64.b64decode('eNrNXFtvHMl5/SsNPcVeSan6LnVRHoKFEycBAj/ED34wDKOu0mB5C4dceR3kv+fUDFekSGm6OxwOCGO9y5nisOY7VefyVXf/+X/e/LS5qG8+TG/S9ebm03m72ZQ3b6c3F7fnuV1v8caf/4Ifr64v81k7/2vfXHxs11fXm4ub8TtskvRm1LCnFBPbLlbYRlNtiibV5KyJiUIx+LGLccK+SvGBUi8muDcPPnqzm8X19m/vfqYPn9pZvby9+XA/qQ/GmLvh51e7P/6ny+ufJgyabj61qf0tlZsJs27Xm5LOpuu2vT27mfrlNd7ebCe8Vm7P0s3m8uLD9Lu7H9ruV7e359Nln0jjlC7qJOH93d+57Pvv6MfP26uzze7P3k1tvHaTrj+28SLGjB/b+dX41Aej3t1/gXfmzf++nZ5Xb4nahXpMwVrOWUNqjgun0qJjVc8+18Q9mlB99lpNydSst52rJ6kr621fpt74xHqLX0bNrfO7mgf9uuZWbNTZqu9GzdfdPr/uVa0LoQY2NghXp02D88kqFrIroeBfiSlVl1Px3fjOTnLIpucKuKquqzsdq+6/x1fe/Vbd9N6u20Vp0+dP7WIKYcLY7W2+ucbHtTp1/DXUk76GQWUWA5V5AOj5AFhqagv+Ca4JFjLWdAafsOuWXOk1dpONeJNTy1KcGtTd+GTYGE+G+joA+FgA/Mvm501tkzWOzJR/me4o5uPm5/1W+O/by5tNu7h5tPqDmV/7wcwXnp9f+ORj0I6aa/M1BZYs4HzlHCo2gS/EwVjbI1eDTWK88xihxUfyrFFkXeHlRRneitmV37uv603WzdYbY07C8F4kNmMae7ZY3KkVlNFG71yyo+KmdMMMGLKpzYhTct3UzoVKKrnGdfXWF2d4CryvuT6uOZEuqDpGnYThfWnBpqGabJtrWdlFLQkMTjk4k2Mj47RCsAj0HzqWfuZeO+xOzNWEdXV3L87wYJpvMDyJeUw1dgHV2JNwvIpwAa8bDlQlU0idjYtOSNnkrh1vRvgdYVJhSd5p8mR97r3Hpis53h+b4xUTGxx/Z2lmOJ4ozq9+iqfh+ATPAoPuTaQk3IJWtpVrL7VxUQ9+MYGrtGZz4BgL9CCBiZSyh8dP6wofXpbjA+35Jj6qt5tf6BhzEo4HTZsCAS9cJNsqsDQqFKRlgsWBCibWwKnreDnmBHr3pfjMVhtUNq+rd3x5jifdu/jHNYeVXcDxY9RJOF68425MjZUqxdpqtCF0z8WoI0m9+WZS6xpqT5orZ4h+z7FLNxEpat06t+blOV6/zfH0mOP9Ao73p+F4B92ETYR2+mJLDdZBaA1UFKtfWy+UbER6qqVySb5E37y1NgVvJARdZ2+sPTbH2+jCjuPDIh/v530lxpyE44PtWakGZ3Nja0pwoXXKWmByqsTkJIzt4LMksSVoptZbbolqI0/drCs8vSzHO9nzzaNODSnNs43SSTie4Ukk1xogmxosbDCb1oxzLudgvVGDzOS1+yjQUW4edqYJyi9ka+51Xb355Tne7m2NC487NV4XrPIx6iQc31trsdTaHRZ7lmpaT0RpBCoTegKpc7LcQTdFQDTig1SXucTUg4HYrqu7vDjHy3c4/onUygKhPU2vprBvWMe+ZjFZHHQW5ibUjMXeQ+81s4PXcWJj7ZoNEi3yVFCfm/TR41kHgR6Z4wN72rVqaBHFL7CVdomtPALFZ85Ggxm1bzkK1nntrVebfOqFsxdwDMck2Vtil4Vcjd6VyDaST184e2Hd3ctSfPTfbhv4BRTvT0PxlcgXciaVkmEZY7G+UoKxISrJDa+ZbbdqsiHvbIKd1BZgO+H8I7Ba6WX8CVo1bldzfdwjEDYLGpJj1EkovhGxhAifSLGU7rJBSamY0GIr2GwWhGPhWkz3KRYFxQQbQ9LgjDRbV9r48OIU79w3m/HxMQw0T/EYcxKKdyKZ1cdkcmHn/bAyyXQ4nJYhsrYQLEzqtRSIsUj3rYzTKeVWFazf1kEQj03xFOOgeOFFFB95vu6RV1N8uby9uKmXny8elV2hFoqdJPEAAAYeGws9xj5wyNJR74zCBiUfvIeuajO9h5RBRtl1WKBEIcOJMswozXP9l9k9PXf93a9vTVe3f//7Wfsw3W4xu+l+4ntUzn6ZLrHI3075dnO2X/r7gmEExnxGYaYf3k7v3k6/fTv94w6Kq4R9gYHbtn0//fHT5ecpTT+ns03FJ14B2i1gvPtrLZVPO+gupn3xvvqj76d/+xXVu7+ZttPv/+MPP/7nX3/8wx//9K//9TXG/6B2+gGT/830W0xuFu4x/+/g/aVujxXme3AzVg5DsNgecrXZ5UhOxTfqduy4bhElTCUovLddaw/UikOQqNiNxnsWIgQQ1QYqTKvgtsvgvp/4HNzWBP+q4GYGzkyAnBfYN/Pd3sg92HYZ2GPZCD6N/aEeie+kbvQIpZLNHbrcimnJOR8yzI3RklMJbFPyrcYYPXnukF8ETSBvVoFNy8C+n/gc2ETQs9cENqgGm9pP7zD5BS1352bBpoVEDi7k8HZycgBsqrB2qWl2sJFpHPtFKUB35KcabQPVcLfAP8dssesbs0sOJM61+hx1Fdi8kMi/THx2ZyMlvS4eN2NTh9/g/90Cu/TdZtA92LycxgVLR8yhqxgsp1CyxKxSyzi1cnBImrrtHSqNzd7GmW8xmgXWNZtenXpfTSJDwa7b2bKcxvcTnwObrTOvC23w+A+Y/E615xPKmP+xVFsU2wOLR/UQkbcUe0+pIKsARx3HCmy9Zuu0VIi3D+yChS2LhdVmGn65CHGHrU7JroJbFxL5l4nPEnkQfV1ErsDZDdXWBadKQfRYqu14z4dyyKJlIcROpE+JJOS6Bo3QbmxsKjGVmjKLCWlcJUm2c+idTJCSK8PSZfGrwHbLwL6f+CzYyGavCmy3s2gBqi0LDm4x+6OpNlIM67C2h1qcYHGE3Z4h1qW5GELS2Ea6LVp8l2akMQFzuPXoxSL6ZrUJPwX8Zy2rwPbL49d+4rOqbcMrTF+sQ7V5vr+H6R9LtcXs2fCgHzc9F4nBQrT7iN21a0TCRpwelwC13jx8+Wi6Vki7OKM9ZJ/ZWMW/ucVVYIeFNP5l4rOq7b28LrRleDRHQ7UXOPIx/6Nl7dGhgNdROhi/ELgiHHcxRIAx5xAF2SolxjsR6atU31CSnChG1wv+J7V541zj3nkV3HGhSfsy8dm97cPr2txsgbO6odoLtjZmf8ysvavbIUduQuYcWq4KTcZmdcnnsiP2VpG4sJNtFIfYHb2zJjm49tyBsm3aqmG3Bmxrlmft/cRnwRYXX1/WBuLvMPkFJwYuHku1BVwow+kcytpqYmu1l+AbtrhHuk695dgreJ3UIXEHKylbcuRK7KkFbU5rro58dLQqftmFXbT7ic+Cza+Nx4cVl13WlgVZm+VoWXtkGHyaHuqQs+QOM67qUqIMIeY6zqVDdyEic5cqlFwOiZuJbBobcb3DzXWfsol+lWrbhV20+4nPga0Bnv11ZW3d9VSGaut8h3zM/1iqrYMP/Yxqd5+dq7bU2pDCxCgHV03B9g5Wu0P4xgu5RCqBkLYScQm2eWh2CznkVR1yu7SP9mXi801Tfl1ZW3dE7pep9pj90bI2PmncU8Th0IWcYmPM3vhxLZXBprXUmwmmldjcuHZ58HyTxAYGzTssDdudth5ZNcPLrQJ7YR/tfuKzjlz86+qautFYURkd8jDvx8XTsVSb7/oTB8++KEdws2RtSqI5UPO5w28LV0XW4m5zTsaNu02JumB4qU4zTFxLVf2qxorVFWdfYdnZF73Cnum+Q77o8IuOqdo7NjzYMo2+wW6l1o3IAFGRpAPDfuFna7XkyN1FX3IN6h0Xz44Nq1cwwn2rZBnYbrlq7yc+u7Pjq+uQD9Vmv1Pt+abpmP/ROuTxrvfIh+664nGvbAsEPY7Y0S0zGYNwLdFw6W3cDtEpl4x97lyo3EkEio1FEoaHXwW3XxG/dhOf3dvR6CuMX2EY8iXXrJijqfbotbODuT24t7m4LCkYTkWSZXLZIXVZLjErWRu6cVEQu2u1gSNCdpZmk23EsPFmnSMPy49D9hOfP+rUV3gcwg6q7RbcKo/ZHy1rj+sAIIDukGqXce2vRbzqQD0G+GzLrVqbSgvBFABryUiwLsVxAi7N5WpcyrmEKuzDKrAXdtHuJz7fWHll6UuGFWfenWvbJQ9GWKva57dnN5vtTbtafhFgzCaVCOOdUnW9ZnHZtsbBDC637MdtVSDrnkDdCQwfuYpGKLh0+DU7f7z5ZU5Pr0H7cTq7/Ij3NmU7fbrN0xbf/Wa7x2w0SgBUaWcAycbp5vq2/LSdajtDya8n6399d4fR2wHCxcNXgfG0BcxT+px+eT/9O1A+Txe/TNftPG0u/vnrTbjg3Fnke6L65Qs+FtX/Bxrel3EXGxdRqdwMaeOWgg3UY66jqWESrLMzZJ2LtLu5OXvMLQOvEOsqNB41t35ErVDmdLG9ury+QaWvLndXZ6KQo5iWdNqOrTNKfANQxhWc+EbYKeO+2f2r2Bb7PTOuRGZsSeyx3Z67hgsYeyXgv1DB0s6BzfYBLuODz1q/+RqZd2Tnk+sYNIuNfTY2UDpvE1OLMbBxxRnjvPUdUdWSuJaTT8WNa4+sGwPHaZNS7Iql3IvqKmzo8U75jAJ9uhy1/gSSsVjp5Rpfdvv219JuJ5Lp4/Xl7dV2fwvtlwGj/ttPG7wOSPYvvr+/kBnfMp3dsduj9gHN3xOEMbOlp2eXvvQSTIL81J6S5JpqTA1hEVpUGvkGDHoPhUsqHQIUR8xMu5uEfC59wb3ND0vPj0t/jqrfnu/rznHazarV6ePtpo5KgqhuLm+v90S02w1Ed2/+E35humif737cLfI7Bmv1weK/e/tb3PRO4zwKY9AsDM/XiuIpcRPPmgRfFgG9V9ttqRYuPvjd7eYVhhRuIBcWaruH6LRqerEEVVkFg6zQCmvvtYLcE62wR9QKXvBYFw4n0YqaQ3KNiJCkpFXt0SRvanQI3aKmWlNqs9nWnIWEqq8kHnrf1KW1m0JXaoU/oBXyVCv4GFrBNH8QPQadQCvUZzcuDnQ0TidaMAhKcFZsxFPPraRsYbkaG/AYZBtJqkM+oqEk3Sxpdz3Exs1ohcZvaIV5qBX+2VohC+6SFn8KrWCq4JqCBOpK1NaK6xECzDGI13FDETtNcE/SoufamhuvJTWcfe95wbOOHpbeH9QKfqoV9Fgr4JyOpxXMsmAHfPcM75haIcHAG1B0qfVuXYgB6UEkuU7qWxhnsdUQkqKU8RQkm0xgdcAKRtbXQKtgCCu0QulQrghHlApa8MgGCv4EUjHu3crjiVMZ6qAZcU97AyFJl1iCKuRbxGdSqHbGDrHWjefxlMhcQ49hFRhxnVR4832p8C+kFMoLLBWfQCc8jKuVHqDdsFPeR5HxDLCqo09eyXjsDk/cWzRxXJs2HmOiwMT6iJyYV6Vva2Z0gvmpToCwHuhEfLZOeDt/OOW/e2ngUTOFgSBQVkJQG3fWxZ5sJi94tTW13jdGpLDWmOx0hD6GhtvdaaSaFnlV6e0hnVDzVCfCE51wR9QJGMT5DSDfvfb6mDpRW8+uMplxpRaEuXHW7BxBsWthbUgSWjNFEJcJ46SfAwXrYlPpSVJZBQOt0An/MFPYJ5nimEKhC66XVaYTCEVQn2BZERt0PDPGUi3UpbTYNeTkQqOOCI4UaDSWJNBqwnB2MVbFVlolFI8vwJgTCj2UKeI3+k9yjP5TtAv6T9GeQCtCBCRaYoB/DcEWscZz4mKhDLZ0j1wuPdpESGsdqAnB6VpxSBumNF3lqKzMaIX5Rv+J9YFWPF8qOMYFZ53xBFKRUdHgXKxY98lanzJcUjceDlethg51CIkZo4r38LEwpIaMiOOR+HgdR+khqZCn7Sd60n4CCx1PKuASF2wAOUn7KedUerDJ1WQgxSEjVxuBn/XgHx7PwIJuw8maaGvxIbRI5HtBuKDRP18Fg1tzVOEOSsVRjyqOlLOfLxUJmq0Owdn3UXVmrdQUhtZy1+I8ScwEGYeHHXe7DoPrU6A6uoZG/Kr20+NT/zmpMHxAKr4RKigco/204AR3DDpF+6na7oKDRETJYTz/DVmvSYS6dyDTjId+tOBStDKeC5qFQWk5YKe07hcc0j/EJsxIhaeZWEHm+bEizGsFxpxAKyDGudeg8PwBm8EJolwk2NpixwNuEjXCNlCGdsciGqKxOVUfq+vNLnlE38PSx0NaQd+IFfbJUYU5ZqxY0PAYg5ZoxV/+D6e9Szw=')))
assert digest(CASES)==MANIFEST['heldout_sha256']
BANK={p['problem_id']:p for p in CASES}


PLAN=[[p['problem_id'],phase] for p in CASES for phase in ['source','finalize','remask','regenerate']]
assert len(PLAN)==len(CASES)*4
PROTOCOL={'study':STUDY,'version':MANIFEST['version'],'manifest_sha256':digest(MANIFEST),
         'cases_sha256':digest(CASES),'model_revision':MODEL_REV,'code_commit':CODE_COMMIT,
         'q':Q,'tau':TAU,'window':WINDOW,'plans':PLAN,'budgets':{k:MANIFEST[k] for k in ['source','finalize','regenerate','remask']}}
PROTOCOL_HASH=digest(PROTOCOL)
save_new_json(ROOT/'protocol.json',{'protocol_hash':PROTOCOL_HASH,'protocol':PROTOCOL})
save_new_json(ROOT/'cases.json',CASES)
print('Protocol',PROTOCOL_HASH,'existing',len(read_records()),'/',len(PLAN))


POLICY_PATH=Path('/content/drive/MyDrive/RepairScopeX/V2_14_Calibration/LOCKED_V2_POLICY.json')
if not POLICY_PATH.exists():
    raise RuntimeError('Run V2-14 until all 192 calls complete; frozen policy is missing. No V2-15 trials attempted.')
POLICY=json.loads(POLICY_PATH.read_text())
assert POLICY['policy_sha256']==digest({k:v for k,v in POLICY.items() if k!='policy_sha256'}), 'Policy hash mismatch'
assert POLICY['manifest_hash']==digest(MANIFEST) and POLICY['calibration_cases_hash']==MANIFEST['calibration_sha256']
assert POLICY['training_split']=='calibration' and POLICY['test_set_label_access'] is False
ALLOW_FAILED_CALIBRATION=False  # Override only for a clearly labeled descriptive negative study
if not POLICY['calibration_gate_pass'] and not ALLOW_FAILED_CALIBRATION:
    raise RuntimeError('STOP: revised router failed calibration; held-out GPU run blocked. ' +
                       'Do not force prospective confirmation of a failed calibration hypothesis.')
save_new_json(ROOT/'applied_frozen_policy.json',POLICY)
print('Policy SHA256 verified',POLICY['policy_sha256'])


In [ ]:

sys.path.insert(0,REPO_DIR)
from alodlm.generate import load_decoder
from alodlm.inference import DecodeConfig
MODEL_DIR=snapshot_download(repo_id=MODEL_ID,revision=MODEL_REV)
gc.collect();torch.cuda.empty_cache()
args=SimpleNamespace(model=MODEL_DIR,model_config=None,device='cuda',precision='fp32')
decoder=load_decoder(args);tokenizer=decoder.tokenizer
assert next(decoder.model.parameters()).dtype==torch.float32
print('Pinned FP32 ALoDLM loaded')


In [ ]:

SESSION_MAX_NEW=12
SESSION_MAX_MINUTES=28
start=time.monotonic();new=0;failure=None
for pid,phase in PLAN:
    if new>=SESSION_MAX_NEW or time.monotonic()-start>=SESSION_MAX_MINUTES*60: break
    tid=f'{STUDY}|{pid}|{phase}'
    if existing(tid) is not None: continue
    p=BANK[pid];seed=seed_for(pid);src=existing(f'{STUDY}|{pid}|source')
    if phase!='source' and src is None:raise RuntimeError('Missing source: '+pid)
    try:
        if phase=='source':
            mode='left1';budget=96;prefix=[]
            res=model_call(p['prompt']+PRIOR_PROMPT,[],mode,budget,seed)
        elif phase=='finalize':
            mode='left1';budget=128;prefix=[]
            res=model_call(p['prompt']+DIRECT_PROMPT,[],mode,budget,seed)
        elif phase=='regenerate':
            mode='left1';budget=128;prefix=[]
            res=model_call(p['prompt']+FRESH_PROMPT,[],mode,budget,seed)
        else:
            mode='entropy';ids=src['token_ids']
            if len(ids)<2:
                prefix=[];budget=0
                res=dict(out=src['output'],ids=ids,new_ids=[],generated_tokens=0,loop_token_passes=0,
                         stop_reason='not_executable',wall_seconds=0.,peak_vram_gb=0.)
            else:
                cut=repair_cut(ids,'first_math');prefix=ids[:cut]
                budget=min(96,max(1,len(ids)-cut))
                res=model_call(p['prompt']+PRIOR_PROMPT,prefix,mode,budget,seed)
                assert res['ids'][:len(prefix)]==prefix
        rec={'trial_id':tid,'problem_id':pid,'kind':p['kind'],'split':p['split'],
             'phase':phase,'protocol_hash':PROTOCOL_HASH,'seed':seed,
             'output':res['out'],'token_ids':res['ids'] if phase=='source' else [],
             'stop_reason':res['stop_reason'],'generated_tokens':res['generated_tokens'],
             'loop_token_passes':res['loop_token_passes'],'wall_seconds':res['wall_seconds'],
             'peak_vram_gb':res['peak_vram_gb'],'budget':budget,'decoder':mode,
             'preserved_tokens':len(prefix) if phase=='remask' else 0,
             'score':dual_score(res['out'],p)}
        atomic_record(rec);new+=1
        print('SAVED',new,'/',SESSION_MAX_NEW,phase,pid,'outcome',rec['score']['class'],flush=True)
    except Exception as ex:
        failure=repr(ex)
        (ROOT/'last_error.json').write_text(json.dumps({'trial_id':tid,'error':failure},indent=2))
        torch.cuda.empty_cache()
        print('STOP: checkpoint preserved',failure)
        break
print('Session new:',new,'| total:',len(read_records()),'/',len(PLAN))
if failure:print('Failure details:',ROOT/'last_error.json')


In [ ]:

records=read_records(); by={(r['problem_id'],r['phase']):r for r in records}
complete=[p for p in CASES if all((p['problem_id'],a) in by for a in ['source','finalize','remask','regenerate'])]
rows=[]
for p in complete:
    pid=p['problem_id'];source=by[(pid,'source')]
    row={'problem_id':pid,'kind':p['kind'],'bucket':bucket_of(source,p['kind'])}
    for a in ['source','finalize','remask','regenerate']:
        rr=by[(pid,a)]; row[a+'_correct']=int(rr['score']['complete_correct']); row[a+'_class']=rr['score']['class']
        row[a+'_passes']=rr['loop_token_passes'];row[a+'_seconds']=rr['wall_seconds']
    rows.append(row)
df=pd.DataFrame(rows)
if len(df): display(df.groupby('kind')[[a+'_correct' for a in ['source','finalize','remask','regenerate']]].agg(['count','sum']))
print('Complete problems',len(complete),'/',len(CASES),' saved calls:',len(records))
status={'complete_problems':len(complete),'planned_problems':len(CASES),'calls':len(records),'planned_calls':len(PLAN)}
(ROOT/'progress.json').write_text(json.dumps(status,indent=2))
if len(df):df.to_csv(ROOT/'paired_summary.csv',index=False)
snapshot_export()


In [ ]:

if len(complete)!=len(CASES):
    print('Independent confirmation INCOMPLETE. Saved records remain intact. Do not interpret interim held-out outcomes.')
else:
    from scipy.stats import binomtest
    from collections import defaultdict
    pairs=[]
    for p in CASES:
        x=next(r for r in rows if r['problem_id']==p['problem_id'])
        source=by[(p['problem_id'],'source')]
        selected=action_of(POLICY,source,p['kind'])
        chosen='source' if selected=='preserve' else selected
        y=int(x[chosen+'_correct']); source_ok=int(x['source_correct'])
        pairs.append({'problem_id':p['problem_id'],'kind':p['kind'],'router_action':selected,'router_correct':y,
                      'source_correct':source_ok,'finalize_correct':int(x['finalize_correct']),
                      'remask_correct':int(x['remask_correct']),'regenerate_correct':int(x['regenerate_correct']),
                      'router_additional_passes':0 if chosen=='source' else x[chosen+'_passes'],
                      'router_total_passes':x['source_passes']+(0 if chosen=='source' else x[chosen+'_passes']),
                      'source_passes':x['source_passes']})
    table=pd.DataFrame(pairs)
    table.to_csv(ROOT/'FINAL_LOCKED_PROSPECTIVE_RESULTS.csv',index=False)
    metrics={}
    for comparator in ['source','finalize','remask','regenerate']:
        ref=table[comparator+'_correct'].to_numpy();out=table['router_correct'].to_numpy()
        win=int(((out==1)&(ref==0)).sum());lose=int(((out==0)&(ref==1)).sum())
        exact=float(binomtest(min(win,lose),n=win+lose,p=.5,alternative='two-sided').pvalue) if win+lose else 1.0
        metrics[comparator]={'router_accuracy':float(out.mean()),'baseline_accuracy':float(ref.mean()),
                  'wins':win,'losses':lose,'paired_exact_p':exact,'difference':float(out.mean()-ref.mean())}
    rng=np.random.default_rng(20261024)
    changes=table['router_correct'].to_numpy()-table['source_correct'].to_numpy()
    boots=[float(np.mean(changes[rng.integers(0,len(changes),len(changes))])) for _ in range(10000)]
    metrics['bootstrap_95_ci_delta_vs_preserve']=np.quantile(boots,[0.025,.975]).tolist()
    metrics['by_kind']=table.groupby('kind')[['source_correct','router_correct','finalize_correct','remask_correct','regenerate_correct']].mean().to_dict(orient='index')
    metrics['router_action_distribution']=table['router_action'].value_counts().to_dict()
    metrics['mean_router_total_passes']=float(table['router_total_passes'].mean())
    metrics['mean_source_passes']=float(table['source_passes'].mean())
    metrics['protocol_hash']=PROTOCOL_HASH
    metrics['frozen_policy_sha256']=POLICY['policy_sha256']
    save_new_json(ROOT/'FINAL_PROSPECTIVE_STATISTICS.json',metrics)
    print(json.dumps(metrics,indent=2))
    print('ALL TEST RESULTS COMPLETE. This is independent confirmation on a new synthetic cohort, not external benchmark generalization.')
    snapshot_export()
